# Notebook 00 — Descarga de Datos Crudos

## Objetivo

Este notebook descarga los tres datasets crudos del proyecto y los guarda en `data/raw/`.

**Idempotencia**: `download_sp500()` y `download_macro()` verifican si el CSV ya existe
en `data/raw/` antes de pegarle a la API. Si ya está descargado, simplemente lo cargan.
Esto significa que correr este notebook de nuevo **no vuelve a descargar nada** a menos
que borres los archivos de `data/raw/` manualmente.

Este notebook se corre **una sola vez** (o si se necesita regenerar `data/raw/` desde cero).
El notebook `01_data_pipeline.ipynb` asume que estos archivos ya existen.

In [1]:
import sys
import os
sys.path.insert(0, "..")

from src import data_loader, utils

utils.set_plot_style()

RAW_DIR = "../data/raw"
os.makedirs(RAW_DIR, exist_ok=True)

START = "2008-01-01"
END   = "2025-12-20"

In [2]:
news_check = data_loader.load_news(RAW_DIR)
print(f"Noticias desde:    {news_check['Date'].min().date()}")
print(f"Noticias hasta:    {news_check['Date'].max().date()}")
print(f"Total de noticias: {len(news_check):,}")

Noticias desde:    2008-01-02
Noticias hasta:    2025-12-18
Total de noticias: 27,212


## Descarga de precios S&P 500

Ticker `^GSPC` via `yfinance`. Guarda en `data/raw/sp500_raw.csv`.

In [8]:
sp500_path = os.path.join(RAW_DIR, "sp500_raw.csv")
sp500 = data_loader.download_sp500(START, END, sp500_path)

print(f"S&P 500: {sp500.shape} | {sp500.index[0].date()} → {sp500.index[-1].date()}")

[data_loader] Descargando ^GSPC de yfinance (2008-01-01 → 2025-12-20)...


[*********************100%***********************]  1 of 1 completed

[data_loader] Guardado en ../data/raw/sp500_raw.csv (4,522 filas).
S&P 500: (4522, 5) | 2008-01-02 → 2025-12-19


## Descarga de indicadores macroeconómicos (FRED)

Series descargadas: VIX (`VIXCLS`), spread de tasas (`T10Y2Y`), Fed Funds Rate
(`FEDFUNDS`), inflación (`CPIAUCSL`) y desempleo (`UNRATE`).

La API key se lee de `.env` dentro de `download_macro()` — no se expone en el notebook.
Guarda en `data/raw/macro_fred.csv`.

In [10]:
macro_path = os.path.join(RAW_DIR, "macro_fred.csv")
macro = data_loader.download_macro(START, END, macro_path)

print(f"Macro FRED: {macro.shape} | {macro.index[0].date()} → {macro.index[-1].date()}")

[data_loader] Descargando indicadores de FRED (2008-01-01 → 2025-12-20)...
[data_loader] Guardado en ../data/raw/macro_fred.csv (4,750 filas).
Macro FRED: (4750, 5) | 2008-01-01 → 2025-12-19


## Noticias financieras (Kaggle)

Este dataset **no se descarga via API** — es un CSV estático de Kaggle (19,127 headlines,
2008–2024) que debe colocarse manualmente en `data/raw/sp500_news.csv`. Por eso no existe
un `download_news()` en `data_loader.py`, solo `load_news()`.

Si el archivo no está presente, hay que bajarlo de Kaggle antes de continuar.

In [ ]:
news_path = os.path.join(RAW_DIR, "sp500_news.csv")
if not os.path.exists(news_path):
    raise FileNotFoundError(
        f"No se encontró {news_path}.\n"
        "Este dataset viene de Kaggle y no se descarga via API: "
        "hay que colocarlo manualmente en data/raw/sp500_news.csv."
    )

news = data_loader.load_news(RAW_DIR)
print(f"Noticias: {news.shape} | {news['Date'].min().date()} → {news['Date'].max().date()}")

## Resumen estadístico de los datos crudos

Verificación rápida de tipos de datos, nulos y shape de los tres datasets descargados.

In [ ]:
utils.dataset_summary("S&P 500 Precios", sp500)
utils.dataset_summary("Macro FRED", macro)
utils.dataset_summary("Noticias Financieras", news)

## Siguiente paso

Con los tres datasets en `data/raw/`, el pipeline de features continúa en
`01_data_pipeline.ipynb`, que calcula los indicadores técnicos, integra el macro y
define la variable objetivo.